# Open Source Contribution Maximizer

This notebook implements an intelligent system for maximizing open source contributions using Celery for distributed task processing, GitHub's API for repository analysis, and ML-powered matching algorithms. The agent helps developers find meaningful projects to contribute to, tracks contribution opportunities, and optimizes impact across the open source ecosystem.

In [ ]:
# Install required packages
!pip install celery redis PyGithub requests beautifulsoup4 pandas numpy scikit-learn
!pip install gitpython python-dateutil aiohttp asyncio
!pip install sentence-transformers transformers torch

In [ ]:
import os
import asyncio
import aiohttp
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Any, Set
from dataclasses import dataclass, field
from enum import Enum
import json
import re
import time
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# GitHub API
try:
    from github import Github, GithubException
    import git
except ImportError:
    print("GitHub libraries not available. Using mock implementations.")

# Task queue
try:
    from celery import Celery
    from celery.result import AsyncResult
except ImportError:
    print("Celery not available. Using synchronous processing.")

# ML libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

# NLP for semantic matching
try:
    from sentence_transformers import SentenceTransformer
    from transformers import pipeline
except ImportError:
    print("Advanced NLP libraries not available. Using basic text processing.")

# Web scraping
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import dateutil.parser

In [ ]:
class ContributionType(Enum):
    """Types of open source contributions"""
    CODE = "code"
    DOCUMENTATION = "documentation"
    BUG_REPORT = "bug_report"
    FEATURE_REQUEST = "feature_request"
    REVIEW = "review"
    TRANSLATION = "translation"
    DESIGN = "design"
    TESTING = "testing"
    COMMUNITY = "community"
    MENTORING = "mentoring"

class DifficultyLevel(Enum):
    """Difficulty levels for contributions"""
    BEGINNER = "beginner"
    INTERMEDIATE = "intermediate"
    ADVANCED = "advanced"
    EXPERT = "expert"

@dataclass
class DeveloperProfile:
    """Developer skill profile and preferences"""
    username: str
    programming_languages: List[str]
    frameworks: List[str]
    experience_level: str
    interests: List[str]
    contribution_types: List[ContributionType]
    time_availability: int  # hours per week
    preferred_project_size: str  # small, medium, large
    github_stats: Dict[str, Any] = field(default_factory=dict)
    past_contributions: List[Dict[str, Any]] = field(default_factory=list)
    skills_vector: Optional[np.ndarray] = None

@dataclass
class Repository:
    """Repository information"""
    name: str
    full_name: str
    url: str
    description: str
    language: str
    stars: int
    forks: int
    open_issues: int
    contributors: int
    last_update: datetime
    topics: List[str] = field(default_factory=list)
    good_first_issues: List[Dict[str, Any]] = field(default_factory=list)
    help_wanted_issues: List[Dict[str, Any]] = field(default_factory=list)
    contribution_guidelines: Optional[str] = None
    maintainer_responsiveness: float = 0.0
    activity_score: float = 0.0
    newcomer_friendliness: float = 0.0
    technologies: List[str] = field(default_factory=list)

@dataclass
class ContributionOpportunity:
    """Specific contribution opportunity"""
    repository: Repository
    issue_number: Optional[int]
    title: str
    description: str
    contribution_type: ContributionType
    difficulty: DifficultyLevel
    estimated_hours: int
    labels: List[str]
    match_score: float
    created_at: datetime
    last_activity: datetime
    mentor_available: bool = False
    impact_potential: float = 0.0
    learning_value: float = 0.0
    community_engagement: float = 0.0

@dataclass
class ContributionPlan:
    """Personalized contribution plan"""
    developer: DeveloperProfile
    opportunities: List[ContributionOpportunity]
    timeline: Dict[str, List[ContributionOpportunity]]
    learning_goals: List[str]
    skill_development_path: List[str]
    estimated_impact: Dict[str, float]
    success_metrics: Dict[str, Any]

In [ ]:
class GitHubAnalyzer:
    """Analyze GitHub repositories and opportunities"""
    
    def __init__(self, github_token: Optional[str] = None):
        self.github_token = github_token or os.getenv('GITHUB_TOKEN')
        self.github = Github(self.github_token) if self.github_token else None
        self.session = requests.Session()
        
        if self.github_token:
            self.session.headers.update({
                'Authorization': f'token {self.github_token}',
                'Accept': 'application/vnd.github.v3+json'
            })
    
    def search_repositories(self, query: str, language: Optional[str] = None, 
                          min_stars: int = 10, max_results: int = 100) -> List[Repository]:
        """Search for repositories based on criteria"""
        if not self.github:
            return self._mock_repositories()
            
        repositories = []
        
        try:
            # Build search query
            search_query = f"{query} stars:>={min_stars}"
            if language:
                search_query += f" language:{language}"
            
            # Add labels for beginner-friendly projects
            search_query += ' "good first issue" OR "help wanted" OR "beginner friendly"'
            
            repos = self.github.search_repositories(
                query=search_query,
                sort='stars',
                order='desc'
            )
            
            for repo in repos[:max_results]:
                try:
                    repository = self._analyze_repository(repo)
                    if repository:
                        repositories.append(repository)
                except Exception as e:
                    print(f"Error analyzing repository {repo.full_name}: {e}")
                    continue
                    
        except GithubException as e:
            print(f"GitHub API error: {e}")
            return self._mock_repositories()
        
        return repositories
    
    def _analyze_repository(self, repo) -> Optional[Repository]:
        """Analyze a single repository"""
        try:
            # Get basic info
            repository = Repository(
                name=repo.name,
                full_name=repo.full_name,
                url=repo.html_url,
                description=repo.description or "",
                language=repo.language or "Unknown",
                stars=repo.stargazers_count,
                forks=repo.forks_count,
                open_issues=repo.open_issues_count,
                contributors=self._get_contributor_count(repo),
                last_update=repo.updated_at,
                topics=list(repo.get_topics())
            )
            
            # Analyze issues for contribution opportunities
            repository.good_first_issues = self._get_labeled_issues(repo, 'good first issue')
            repository.help_wanted_issues = self._get_labeled_issues(repo, 'help wanted')
            
            # Get contribution guidelines
            repository.contribution_guidelines = self._get_contribution_guidelines(repo)
            
            # Calculate metrics
            repository.maintainer_responsiveness = self._calculate_responsiveness(repo)
            repository.activity_score = self._calculate_activity_score(repo)
            repository.newcomer_friendliness = self._calculate_newcomer_friendliness(repo)
            repository.technologies = self._extract_technologies(repo)
            
            return repository
            
        except Exception as e:
            print(f"Error analyzing repository: {e}")
            return None
    
    def _get_contributor_count(self, repo) -> int:
        """Get number of contributors"""
        try:
            contributors = list(repo.get_contributors())
            return len(contributors)
        except:
            return 0
    
    def _get_labeled_issues(self, repo, label: str) -> List[Dict[str, Any]]:
        """Get issues with specific label"""
        issues = []
        try:
            for issue in repo.get_issues(labels=[label], state='open')[:10]:
                issues.append({
                    'number': issue.number,
                    'title': issue.title,
                    'body': issue.body or "",
                    'labels': [l.name for l in issue.labels],
                    'created_at': issue.created_at,
                    'updated_at': issue.updated_at,
                    'comments': issue.comments,
                    'assignees': [a.login for a in issue.assignees],
                    'url': issue.html_url
                })
        except Exception as e:
            print(f"Error fetching issues: {e}")
        
        return issues
    
    def _get_contribution_guidelines(self, repo) -> Optional[str]:
        """Get contribution guidelines"""
        try:
            # Check for CONTRIBUTING.md
            for filename in ['CONTRIBUTING.md', 'CONTRIBUTING.rst', 'contributing.md', '.github/CONTRIBUTING.md']:
                try:
                    content = repo.get_contents(filename)
                    return content.decoded_content.decode('utf-8')[:1000]  # First 1000 chars
                except:
                    continue
        except:
            pass
        return None
    
    def _calculate_responsiveness(self, repo) -> float:
        """Calculate maintainer responsiveness score"""
        try:
            # Get recent issues and PRs to check response times
            recent_issues = list(repo.get_issues(state='closed', sort='updated')[:20])
            
            response_times = []
            for issue in recent_issues:
                comments = list(issue.get_comments())
                if comments:
                    first_response = comments[0].created_at
                    response_time = (first_response - issue.created_at).days
                    response_times.append(response_time)
            
            if response_times:
                avg_response_time = np.mean(response_times)
                # Convert to score (lower response time = higher score)
                return max(0, 1 - (avg_response_time / 30))  # 30 days max
        except:
            pass
        
        return 0.5  # Default score
    
    def _calculate_activity_score(self, repo) -> float:
        """Calculate repository activity score"""
        try:
            # Consider commits, issues, PRs in last 30 days
            since = datetime.now() - timedelta(days=30)
            
            commits = list(repo.get_commits(since=since))
            issues = list(repo.get_issues(since=since))
            pulls = list(repo.get_pulls(state='all'))
            
            activity = len(commits) + len(issues) + len(pulls)
            
            # Normalize to 0-1 scale
            return min(1.0, activity / 50)  # 50 activities = max score
        except:
            pass
        
        return 0.3  # Default score
    
    def _calculate_newcomer_friendliness(self, repo) -> float:
        """Calculate how friendly the project is to newcomers"""
        score = 0.0
        
        try:
            # Check for README
            try:
                repo.get_readme()
                score += 0.2
            except:
                pass
            
            # Check for contribution guidelines
            if self._get_contribution_guidelines(repo):
                score += 0.2
            
            # Check for issues with beginner-friendly labels
            good_first_issues = len(self._get_labeled_issues(repo, 'good first issue'))
            help_wanted_issues = len(self._get_labeled_issues(repo, 'help wanted'))
            
            if good_first_issues > 0:
                score += 0.3
            if help_wanted_issues > 0:
                score += 0.2
            
            # Check for documentation
            try:
                contents = repo.get_contents("")
                has_docs = any('doc' in f.name.lower() for f in contents if f.type == 'dir')
                if has_docs:
                    score += 0.1
            except:
                pass
                
        except Exception as e:
            print(f"Error calculating newcomer friendliness: {e}")
        
        return min(1.0, score)
    
    def _extract_technologies(self, repo) -> List[str]:
        """Extract technologies used in the repository"""
        technologies = []
        
        try:
            # Add primary language
            if repo.language:
                technologies.append(repo.language)
            
            # Add languages from GitHub's detection
            languages = repo.get_languages()
            technologies.extend(languages.keys())
            
            # Add topics as potential technologies
            topics = list(repo.get_topics())
            technologies.extend(topics)
            
            # Check for common config files
            config_files = {
                'package.json': ['Node.js', 'JavaScript'],
                'requirements.txt': ['Python'],
                'Gemfile': ['Ruby'],
                'pom.xml': ['Java', 'Maven'],
                'build.gradle': ['Java', 'Gradle'],
                'Cargo.toml': ['Rust'],
                'go.mod': ['Go'],
                'composer.json': ['PHP']
            }
            
            for filename, techs in config_files.items():
                try:
                    repo.get_contents(filename)
                    technologies.extend(techs)
                except:
                    continue
                    
        except Exception as e:
            print(f"Error extracting technologies: {e}")
        
        return list(set(technologies))  # Remove duplicates
    
    def _mock_repositories(self) -> List[Repository]:
        """Mock repositories for demo purposes"""
        return [
            Repository(
                name="awesome-python",
                full_name="vinta/awesome-python",
                url="https://github.com/vinta/awesome-python",
                description="A curated list of awesome Python frameworks, libraries, software and resources",
                language="Python",
                stars=180000,
                forks=24000,
                open_issues=15,
                contributors=450,
                last_update=datetime.now() - timedelta(days=2),
                topics=["python", "awesome", "list"],
                good_first_issues=[
                    {
                        'number': 123,
                        'title': 'Add new machine learning library',
                        'body': 'We should add XYZ library to our ML section',
                        'labels': ['good first issue', 'documentation'],
                        'created_at': datetime.now() - timedelta(days=3),
                        'updated_at': datetime.now() - timedelta(days=1),
                        'comments': 2,
                        'assignees': [],
                        'url': 'https://github.com/vinta/awesome-python/issues/123'
                    }
                ],
                maintainer_responsiveness=0.9,
                activity_score=0.8,
                newcomer_friendliness=0.95,
                technologies=["Python", "Documentation"]
            ),
            Repository(
                name="scikit-learn",
                full_name="scikit-learn/scikit-learn",
                url="https://github.com/scikit-learn/scikit-learn",
                description="Machine learning library for Python",
                language="Python",
                stars=55000,
                forks=24000,
                open_issues=2100,
                contributors=2500,
                last_update=datetime.now() - timedelta(hours=6),
                topics=["machine-learning", "python", "scikit-learn"],
                help_wanted_issues=[
                    {
                        'number': 456,
                        'title': 'Improve documentation for new estimator',
                        'body': 'The new XGBoost wrapper needs better examples',
                        'labels': ['help wanted', 'documentation'],
                        'created_at': datetime.now() - timedelta(days=5),
                        'updated_at': datetime.now() - timedelta(days=2),
                        'comments': 5,
                        'assignees': [],
                        'url': 'https://github.com/scikit-learn/scikit-learn/issues/456'
                    }
                ],
                maintainer_responsiveness=0.85,
                activity_score=0.95,
                newcomer_friendliness=0.75,
                technologies=["Python", "Machine Learning", "NumPy", "SciPy"]
            )
        ]
    
    def analyze_developer_profile(self, username: str) -> Optional[DeveloperProfile]:
        """Analyze a developer's GitHub profile"""
        if not self.github:
            return self._mock_developer_profile(username)
            
        try:
            user = self.github.get_user(username)
            
            # Get repositories and analyze languages
            repos = list(user.get_repos(type='owner', sort='updated')[:50])
            languages = Counter()
            frameworks = set()
            
            for repo in repos:
                if repo.language:
                    languages[repo.language] += 1
                
                # Extract frameworks from repo names and descriptions
                text = f"{repo.name} {repo.description or ''}".lower()
                framework_keywords = {
                    'react', 'vue', 'angular', 'django', 'flask', 'express',
                    'spring', 'rails', 'laravel', 'tensorflow', 'pytorch',
                    'scikit-learn', 'pandas', 'numpy'
                }
                
                for framework in framework_keywords:
                    if framework in text:
                        frameworks.add(framework)
            
            # Determine experience level based on account age and activity
            account_age = (datetime.now() - user.created_at.replace(tzinfo=None)).days / 365
            public_repos = user.public_repos
            
            if account_age < 1 or public_repos < 5:
                experience_level = "beginner"
            elif account_age < 3 or public_repos < 20:
                experience_level = "intermediate"
            elif account_age < 5 or public_repos < 50:
                experience_level = "advanced"
            else:
                experience_level = "expert"
            
            profile = DeveloperProfile(
                username=username,
                programming_languages=list(languages.keys())[:10],
                frameworks=list(frameworks),
                experience_level=experience_level,
                interests=[],  # Would need more analysis
                contribution_types=[ContributionType.CODE],  # Default
                time_availability=10,  # Default
                preferred_project_size="medium",  # Default
                github_stats={
                    'public_repos': user.public_repos,
                    'followers': user.followers,
                    'following': user.following,
                    'account_age_years': account_age
                }
            )
            
            return profile
            
        except GithubException as e:
            print(f"Error analyzing developer profile: {e}")
            return self._mock_developer_profile(username)
    
    def _mock_developer_profile(self, username: str) -> DeveloperProfile:
        """Mock developer profile for demo"""
        return DeveloperProfile(
            username=username,
            programming_languages=["Python", "JavaScript", "Java"],
            frameworks=["React", "Django", "Flask"],
            experience_level="intermediate",
            interests=["machine learning", "web development", "open source"],
            contribution_types=[ContributionType.CODE, ContributionType.DOCUMENTATION],
            time_availability=15,
            preferred_project_size="medium",
            github_stats={
                'public_repos': 25,
                'followers': 50,
                'following': 100,
                'account_age_years': 3.5
            }
        )

In [ ]:
class OpportunityMatcher:
    """Match developers with contribution opportunities using ML"""
    
    def __init__(self):
        self.tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
        self.sentence_model = None
        
        # Try to load sentence transformer for semantic matching
        try:
            self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
        except:
            print("Sentence transformer not available. Using TF-IDF only.")
    
    def find_opportunities(self, developer: DeveloperProfile, 
                          repositories: List[Repository],
                          max_opportunities: int = 20) -> List[ContributionOpportunity]:
        """Find matching contribution opportunities"""
        opportunities = []
        
        for repo in repositories:
            # Create opportunities from issues
            repo_opportunities = self._extract_opportunities_from_repo(repo, developer)
            opportunities.extend(repo_opportunities)
        
        # Calculate match scores
        for opp in opportunities:
            opp.match_score = self._calculate_match_score(developer, opp)
        
        # Sort by match score and return top opportunities
        opportunities.sort(key=lambda x: x.match_score, reverse=True)
        return opportunities[:max_opportunities]
    
    def _extract_opportunities_from_repo(self, repo: Repository, 
                                        developer: DeveloperProfile) -> List[ContributionOpportunity]:
        """Extract opportunities from a repository"""
        opportunities = []
        
        # Process good first issues
        for issue in repo.good_first_issues:
            opp = ContributionOpportunity(
                repository=repo,
                issue_number=issue['number'],
                title=issue['title'],
                description=issue['body'],
                contribution_type=self._classify_contribution_type(issue),
                difficulty=DifficultyLevel.BEGINNER,
                estimated_hours=self._estimate_hours(issue),
                labels=issue['labels'],
                match_score=0.0,
                created_at=issue['created_at'],
                last_activity=issue['updated_at'],
                mentor_available=self._check_mentor_availability(issue),
                impact_potential=self._calculate_impact_potential(repo, issue),
                learning_value=self._calculate_learning_value(issue, developer),
                community_engagement=repo.maintainer_responsiveness
            )
            opportunities.append(opp)
        
        # Process help wanted issues
        for issue in repo.help_wanted_issues:
            opp = ContributionOpportunity(
                repository=repo,
                issue_number=issue['number'],
                title=issue['title'],
                description=issue['body'],
                contribution_type=self._classify_contribution_type(issue),
                difficulty=self._classify_difficulty(issue, developer),
                estimated_hours=self._estimate_hours(issue),
                labels=issue['labels'],
                match_score=0.0,
                created_at=issue['created_at'],
                last_activity=issue['updated_at'],
                mentor_available=self._check_mentor_availability(issue),
                impact_potential=self._calculate_impact_potential(repo, issue),
                learning_value=self._calculate_learning_value(issue, developer),
                community_engagement=repo.maintainer_responsiveness
            )
            opportunities.append(opp)
        
        # Generate repository-level opportunities
        opportunities.extend(self._generate_repo_opportunities(repo, developer))
        
        return opportunities
    
    def _classify_contribution_type(self, issue: Dict[str, Any]) -> ContributionType:
        """Classify the type of contribution based on issue content"""
        text = f"{issue['title']} {issue['body']}".lower()
        labels = [label.lower() for label in issue['labels']]
        
        # Check labels first
        if any(label in ['documentation', 'docs'] for label in labels):
            return ContributionType.DOCUMENTATION
        if any(label in ['bug', 'bugfix'] for label in labels):
            return ContributionType.BUG_REPORT
        if any(label in ['enhancement', 'feature'] for label in labels):
            return ContributionType.FEATURE_REQUEST
        if any(label in ['testing', 'tests'] for label in labels):
            return ContributionType.TESTING
        
        # Check content
        if any(word in text for word in ['documentation', 'readme', 'docs', 'tutorial']):
            return ContributionType.DOCUMENTATION
        if any(word in text for word in ['bug', 'error', 'fix', 'broken']):
            return ContributionType.BUG_REPORT
        if any(word in text for word in ['feature', 'enhancement', 'add', 'implement']):
            return ContributionType.FEATURE_REQUEST
        if any(word in text for word in ['test', 'testing', 'coverage']):
            return ContributionType.TESTING
        if any(word in text for word in ['translation', 'translate', 'i18n']):
            return ContributionType.TRANSLATION
        
        return ContributionType.CODE  # Default
    
    def _classify_difficulty(self, issue: Dict[str, Any], 
                           developer: DeveloperProfile) -> DifficultyLevel:
        """Classify difficulty based on issue content and developer experience"""
        labels = [label.lower() for label in issue['labels']]
        
        # Check explicit difficulty labels
        if any(label in ['good first issue', 'beginner', 'easy'] for label in labels):
            return DifficultyLevel.BEGINNER
        if any(label in ['intermediate', 'medium'] for label in labels):
            return DifficultyLevel.INTERMEDIATE
        if any(label in ['advanced', 'hard', 'expert'] for label in labels):
            return DifficultyLevel.ADVANCED
        
        # Estimate based on content complexity
        text = f"{issue['title']} {issue['body']}".lower()
        complexity_indicators = {
            'expert': ['architecture', 'performance', 'optimization', 'algorithm', 'complex'],
            'advanced': ['refactor', 'redesign', 'implement', 'integration'],
            'intermediate': ['modify', 'improve', 'update', 'enhance'],
            'beginner': ['typo', 'documentation', 'readme', 'comment', 'example']
        }
        
        for level, indicators in complexity_indicators.items():
            if any(indicator in text for indicator in indicators):
                if level == 'expert':
                    return DifficultyLevel.EXPERT
                elif level == 'advanced':
                    return DifficultyLevel.ADVANCED
                elif level == 'intermediate':
                    return DifficultyLevel.INTERMEDIATE
                else:
                    return DifficultyLevel.BEGINNER
        
        # Default based on developer experience
        if developer.experience_level == 'beginner':
            return DifficultyLevel.BEGINNER
        elif developer.experience_level == 'intermediate':
            return DifficultyLevel.INTERMEDIATE
        else:
            return DifficultyLevel.ADVANCED
    
    def _estimate_hours(self, issue: Dict[str, Any]) -> int:
        """Estimate hours required for the contribution"""
        text = f"{issue['title']} {issue['body']}".lower()
        
        # Simple heuristics based on keywords
        if any(word in text for word in ['typo', 'fix typo', 'update link']):
            return 1
        if any(word in text for word in ['documentation', 'readme', 'comment']):
            return 3
        if any(word in text for word in ['bug', 'fix', 'issue']):
            return 5
        if any(word in text for word in ['feature', 'implement', 'add']):
            return 8
        if any(word in text for word in ['refactor', 'redesign', 'architecture']):
            return 20
        
        return 5  # Default estimate
    
    def _check_mentor_availability(self, issue: Dict[str, Any]) -> bool:
        """Check if mentoring is available for this issue"""
        labels = [label.lower() for label in issue['labels']]
        return any(label in ['mentor available', 'mentorship', 'good first issue'] for label in labels)
    
    def _calculate_impact_potential(self, repo: Repository, issue: Dict[str, Any]) -> float:
        """Calculate potential impact of the contribution"""
        # Base impact on repository popularity
        base_impact = min(1.0, repo.stars / 10000)  # Normalize by 10k stars
        
        # Adjust based on issue type
        text = f"{issue['title']} {issue['body']}".lower()
        if any(word in text for word in ['security', 'vulnerability', 'critical']):
            base_impact *= 1.5
        elif any(word in text for word in ['performance', 'optimization']):
            base_impact *= 1.3
        elif any(word in text for word in ['documentation', 'example']):
            base_impact *= 1.1
        
        return min(1.0, base_impact)
    
    def _calculate_learning_value(self, issue: Dict[str, Any], 
                                 developer: DeveloperProfile) -> float:
        """Calculate learning value for the developer"""
        text = f"{issue['title']} {issue['body']}".lower()
        
        learning_value = 0.5  # Base value
        
        # Higher value for technologies the developer wants to learn
        for interest in developer.interests:
            if interest.lower() in text:
                learning_value += 0.2
        
        # Higher value for new technologies
        tech_keywords = ['ai', 'machine learning', 'blockchain', 'cloud', 'kubernetes', 'docker']
        for keyword in tech_keywords:
            if keyword in text:
                learning_value += 0.1
        
        return min(1.0, learning_value)
    
    def _generate_repo_opportunities(self, repo: Repository, 
                                   developer: DeveloperProfile) -> List[ContributionOpportunity]:
        """Generate general repository opportunities"""
        opportunities = []
        
        # Documentation improvement opportunity
        if repo.newcomer_friendliness < 0.8:
            opportunities.append(ContributionOpportunity(
                repository=repo,
                issue_number=None,
                title="Improve documentation and newcomer experience",
                description="Help improve the project's documentation and make it more welcoming to new contributors",
                contribution_type=ContributionType.DOCUMENTATION,
                difficulty=DifficultyLevel.BEGINNER,
                estimated_hours=6,
                labels=['documentation', 'good first issue'],
                match_score=0.0,
                created_at=datetime.now(),
                last_activity=datetime.now(),
                impact_potential=0.7,
                learning_value=0.6,
                community_engagement=repo.maintainer_responsiveness
            ))
        
        # Testing opportunity
        if repo.open_issues > 50:
            opportunities.append(ContributionOpportunity(
                repository=repo,
                issue_number=None,
                title="Add tests to improve code coverage",
                description="Help improve the project's test coverage by writing unit tests",
                contribution_type=ContributionType.TESTING,
                difficulty=DifficultyLevel.INTERMEDIATE,
                estimated_hours=10,
                labels=['testing', 'help wanted'],
                match_score=0.0,
                created_at=datetime.now(),
                last_activity=datetime.now(),
                impact_potential=0.8,
                learning_value=0.7,
                community_engagement=repo.maintainer_responsiveness
            ))
        
        return opportunities
    
    def _calculate_match_score(self, developer: DeveloperProfile, 
                              opportunity: ContributionOpportunity) -> float:
        """Calculate how well an opportunity matches a developer"""
        score = 0.0
        
        # Language match
        if opportunity.repository.language in developer.programming_languages:
            score += 0.3
        
        # Technology match
        tech_overlap = set(opportunity.repository.technologies) & set(developer.frameworks)
        score += 0.2 * min(1.0, len(tech_overlap) / 3)
        
        # Experience level match
        difficulty_exp_match = {
            DifficultyLevel.BEGINNER: ['beginner'],
            DifficultyLevel.INTERMEDIATE: ['beginner', 'intermediate'],
            DifficultyLevel.ADVANCED: ['intermediate', 'advanced'],
            DifficultyLevel.EXPERT: ['advanced', 'expert']
        }
        
        if developer.experience_level in difficulty_exp_match[opportunity.difficulty]:
            score += 0.2
        
        # Contribution type preference
        if opportunity.contribution_type in developer.contribution_types:
            score += 0.1
        
        # Time availability
        if opportunity.estimated_hours <= developer.time_availability:
            score += 0.1
        
        # Repository quality factors
        score += 0.05 * opportunity.repository.maintainer_responsiveness
        score += 0.03 * opportunity.repository.newcomer_friendliness
        score += 0.02 * opportunity.impact_potential
        
        # Semantic similarity (if available)
        if self.sentence_model:
            semantic_score = self._calculate_semantic_similarity(developer, opportunity)
            score += 0.1 * semantic_score
        
        return min(1.0, score)
    
    def _calculate_semantic_similarity(self, developer: DeveloperProfile, 
                                     opportunity: ContributionOpportunity) -> float:
        """Calculate semantic similarity between developer interests and opportunity"""
        try:
            # Create developer profile text
            dev_text = f"{' '.join(developer.interests)} {' '.join(developer.programming_languages)} {' '.join(developer.frameworks)}"
            
            # Create opportunity text
            opp_text = f"{opportunity.title} {opportunity.description} {' '.join(opportunity.repository.topics)}"
            
            # Calculate embeddings
            dev_embedding = self.sentence_model.encode([dev_text])
            opp_embedding = self.sentence_model.encode([opp_text])
            
            # Calculate similarity
            similarity = cosine_similarity(dev_embedding, opp_embedding)[0][0]
            return max(0.0, similarity)
            
        except Exception as e:
            print(f"Error calculating semantic similarity: {e}")
            return 0.0

In [ ]:
class ContributionPlanGenerator:
    """Generate personalized contribution plans"""
    
    def __init__(self):
        self.matcher = OpportunityMatcher()
    
    def generate_plan(self, developer: DeveloperProfile, 
                     opportunities: List[ContributionOpportunity],
                     time_horizon_weeks: int = 12) -> ContributionPlan:
        """Generate a personalized contribution plan"""
        
        # Filter opportunities based on developer preferences
        filtered_opportunities = self._filter_opportunities(developer, opportunities)
        
        # Create timeline
        timeline = self._create_timeline(developer, filtered_opportunities, time_horizon_weeks)
        
        # Generate learning goals
        learning_goals = self._generate_learning_goals(developer, filtered_opportunities)
        
        # Create skill development path
        skill_path = self._create_skill_development_path(developer, filtered_opportunities)
        
        # Estimate impact
        impact = self._estimate_impact(filtered_opportunities)
        
        # Define success metrics
        metrics = self._define_success_metrics(developer, filtered_opportunities)
        
        return ContributionPlan(
            developer=developer,
            opportunities=filtered_opportunities,
            timeline=timeline,
            learning_goals=learning_goals,
            skill_development_path=skill_path,
            estimated_impact=impact,
            success_metrics=metrics
        )
    
    def _filter_opportunities(self, developer: DeveloperProfile, 
                            opportunities: List[ContributionOpportunity]) -> List[ContributionOpportunity]:
        """Filter opportunities based on developer constraints"""
        filtered = []
        
        for opp in opportunities:
            # Check time availability
            if opp.estimated_hours > developer.time_availability * 2:  # Max 2 weeks per contribution
                continue
            
            # Check experience level match
            if developer.experience_level == 'beginner' and opp.difficulty in [DifficultyLevel.ADVANCED, DifficultyLevel.EXPERT]:
                continue
            if developer.experience_level == 'expert' and opp.difficulty == DifficultyLevel.BEGINNER:
                continue
            
            # Check minimum match score
            if opp.match_score < 0.3:
                continue
            
            filtered.append(opp)
        
        return filtered[:15]  # Limit to top 15 opportunities
    
    def _create_timeline(self, developer: DeveloperProfile, 
                        opportunities: List[ContributionOpportunity],
                        time_horizon_weeks: int) -> Dict[str, List[ContributionOpportunity]]:
        """Create a timeline for contributions"""
        timeline = {}
        weekly_hours = developer.time_availability
        
        # Sort opportunities by priority (match score + urgency)
        prioritized_opps = sorted(opportunities, 
                                key=lambda x: (x.match_score, -x.estimated_hours), 
                                reverse=True)
        
        current_week = 1
        week_hours = 0
        
        for opp in prioritized_opps:
            if current_week > time_horizon_weeks:
                break
            
            # Check if opportunity fits in current week
            if week_hours + opp.estimated_hours <= weekly_hours:
                week_key = f"Week {current_week}"
                if week_key not in timeline:
                    timeline[week_key] = []
                timeline[week_key].append(opp)
                week_hours += opp.estimated_hours
            else:
                # Move to next week
                current_week += 1
                week_hours = opp.estimated_hours
                week_key = f"Week {current_week}"
                timeline[week_key] = [opp]
        
        return timeline
    
    def _generate_learning_goals(self, developer: DeveloperProfile, 
                               opportunities: List[ContributionOpportunity]) -> List[str]:
        """Generate learning goals based on opportunities"""
        goals = []
        
        # Technology goals
        technologies = set()
        for opp in opportunities:
            technologies.update(opp.repository.technologies)
        
        new_technologies = technologies - set(developer.programming_languages + developer.frameworks)
        for tech in list(new_technologies)[:3]:
            goals.append(f"Learn {tech} through open source contributions")
        
        # Skill goals based on contribution types
        contribution_types = [opp.contribution_type for opp in opportunities]
        type_counts = Counter(contribution_types)
        
        for contrib_type, count in type_counts.most_common(3):
            if contrib_type == ContributionType.DOCUMENTATION:
                goals.append("Improve technical writing skills")
            elif contrib_type == ContributionType.TESTING:
                goals.append("Master test-driven development")
            elif contrib_type == ContributionType.CODE:
                goals.append("Enhance coding and problem-solving skills")
        
        # Experience-based goals
        if developer.experience_level == 'beginner':
            goals.append("Make first successful open source contribution")
            goals.append("Learn collaborative development workflow")
        elif developer.experience_level == 'intermediate':
            goals.append("Become a regular contributor to 2-3 projects")
            goals.append("Start mentoring other contributors")
        else:
            goals.append("Lead significant feature development")
            goals.append("Become a maintainer of an open source project")
        
        return goals[:5]  # Top 5 goals
    
    def _create_skill_development_path(self, developer: DeveloperProfile, 
                                     opportunities: List[ContributionOpportunity]) -> List[str]:
        """Create a skill development path"""
        path = []
        
        # Start with current skills assessment
        path.append(f"Assessment: {developer.experience_level} level in {', '.join(developer.programming_languages[:3])}")
        
        # Progressive skill building
        if developer.experience_level == 'beginner':
            path.extend([
                "Phase 1: Documentation and small bug fixes",
                "Phase 2: Unit testing and code review",
                "Phase 3: Feature implementation",
                "Phase 4: Code review and mentoring"
            ])
        elif developer.experience_level == 'intermediate':
            path.extend([
                "Phase 1: Complex feature development",
                "Phase 2: Architecture and design contributions",
                "Phase 3: Cross-project collaboration",
                "Phase 4: Community leadership"
            ])
        else:
            path.extend([
                "Phase 1: Technical leadership and mentoring",
                "Phase 2: Project maintenance and governance",
                "Phase 3: Ecosystem development",
                "Phase 4: Industry thought leadership"
            ])
        
        # Add technology-specific milestones
        technologies = set()
        for opp in opportunities[:5]:  # Top 5 opportunities
            technologies.update(opp.repository.technologies)
        
        for tech in list(technologies)[:2]:
            path.append(f"Milestone: Become proficient in {tech}")
        
        return path
    
    def _estimate_impact(self, opportunities: List[ContributionOpportunity]) -> Dict[str, float]:
        """Estimate the potential impact of the contribution plan"""
        if not opportunities:
            return {}
        
        total_impact = sum(opp.impact_potential for opp in opportunities)
        total_learning = sum(opp.learning_value for opp in opportunities)
        total_hours = sum(opp.estimated_hours for opp in opportunities)
        
        # Repository diversity
        unique_repos = len(set(opp.repository.full_name for opp in opportunities))
        
        # Star impact (sum of stars from all repos)
        star_impact = sum(opp.repository.stars for opp in opportunities)
        
        return {
            'total_impact_score': total_impact,
            'learning_value_score': total_learning,
            'time_investment_hours': total_hours,
            'repository_diversity': unique_repos,
            'potential_user_reach': star_impact,
            'average_impact_per_hour': total_impact / total_hours if total_hours > 0 else 0
        }
    
    def _define_success_metrics(self, developer: DeveloperProfile, 
                              opportunities: List[ContributionOpportunity]) -> Dict[str, Any]:
        """Define success metrics for the plan"""
        return {
            'contribution_targets': {
                'total_contributions': len(opportunities),
                'merged_prs': max(1, len(opportunities) // 2),
                'documentation_improvements': len([o for o in opportunities if o.contribution_type == ContributionType.DOCUMENTATION]),
                'code_contributions': len([o for o in opportunities if o.contribution_type == ContributionType.CODE])
            },
            'skill_milestones': {
                'new_technologies_learned': min(3, len(set(tech for opp in opportunities for tech in opp.repository.technologies))),
                'contribution_types_mastered': len(set(opp.contribution_type for opp in opportunities)),
                'mentor_interactions': len([o for o in opportunities if o.mentor_available])
            },
            'community_engagement': {
                'repositories_contributed_to': len(set(opp.repository.full_name for opp in opportunities)),
                'maintainer_connections': len(set(opp.repository.full_name for opp in opportunities)),
                'community_recognition_score': sum(opp.repository.stars for opp in opportunities) / 1000
            },
            'time_management': {
                'planned_hours': sum(opp.estimated_hours for opp in opportunities),
                'weekly_commitment': developer.time_availability,
                'completion_rate_target': 0.8  # 80% completion target
            }
        }

In [ ]:
class OpenSourceContributionMaximizer:
    """Main agent for maximizing open source contributions"""
    
    def __init__(self, github_token: Optional[str] = None):
        self.github_analyzer = GitHubAnalyzer(github_token)
        self.matcher = OpportunityMatcher()
        self.plan_generator = ContributionPlanGenerator()
        
        # Initialize Celery for distributed processing (if available)
        try:
            self.celery_app = Celery('os_maximizer', broker='redis://localhost:6379')
            self.use_celery = True
        except:
            self.use_celery = False
            print("Celery not available. Using synchronous processing.")
    
    def find_contribution_opportunities(self, developer_username: str, 
                                      interests: List[str] = None,
                                      preferred_languages: List[str] = None) -> ContributionPlan:
        """Find and plan contribution opportunities for a developer"""
        
        print(f"🔍 Analyzing developer profile: {developer_username}")
        
        # Analyze developer profile
        developer = self.github_analyzer.analyze_developer_profile(developer_username)
        if not developer:
            raise ValueError(f"Could not analyze developer profile for {developer_username}")
        
        # Override with provided preferences
        if interests:
            developer.interests = interests
        if preferred_languages:
            developer.programming_languages = preferred_languages
        
        print(f"📊 Profile: {developer.experience_level} level, {len(developer.programming_languages)} languages")
        
        # Search for repositories
        print("🔎 Searching for suitable repositories...")
        repositories = self._search_repositories_for_developer(developer)
        print(f"Found {len(repositories)} repositories")
        
        # Find opportunities
        print("🎯 Matching opportunities...")
        opportunities = self.matcher.find_opportunities(developer, repositories)
        print(f"Found {len(opportunities)} contribution opportunities")
        
        # Generate contribution plan
        print("📋 Generating contribution plan...")
        plan = self.plan_generator.generate_plan(developer, opportunities)
        
        return plan
    
    def _search_repositories_for_developer(self, developer: DeveloperProfile) -> List[Repository]:
        """Search repositories suitable for the developer"""
        repositories = []
        
        # Search by programming languages
        for language in developer.programming_languages[:3]:  # Top 3 languages
            try:
                repos = self.github_analyzer.search_repositories(
                    query="good first issue OR help wanted",
                    language=language,
                    min_stars=50,
                    max_results=20
                )
                repositories.extend(repos)
            except Exception as e:
                print(f"Error searching for {language} repositories: {e}")
        
        # Search by interests
        for interest in developer.interests[:2]:  # Top 2 interests
            try:
                repos = self.github_analyzer.search_repositories(
                    query=f"{interest} good first issue",
                    min_stars=20,
                    max_results=15
                )
                repositories.extend(repos)
            except Exception as e:
                print(f"Error searching for {interest} repositories: {e}")
        
        # Remove duplicates
        seen = set()
        unique_repos = []
        for repo in repositories:
            if repo.full_name not in seen:
                seen.add(repo.full_name)
                unique_repos.append(repo)
        
        return unique_repos[:50]  # Limit to 50 repositories
    
    def track_contributions(self, developer_username: str, 
                          plan: ContributionPlan) -> Dict[str, Any]:
        """Track progress on contribution plan"""
        if not self.github_analyzer.github:
            return self._mock_contribution_tracking()
        
        progress = {
            'completed_contributions': [],
            'in_progress_contributions': [],
            'planned_contributions': [],
            'metrics': {
                'completion_rate': 0.0,
                'hours_spent': 0,
                'repositories_contributed_to': 0,
                'prs_merged': 0,
                'issues_closed': 0
            }
        }
        
        try:
            user = self.github_analyzer.github.get_user(developer_username)
            
            # Get recent activity
            events = list(user.get_events()[:100])
            
            plan_repo_names = [opp.repository.full_name for opp in plan.opportunities]
            
            for event in events:
                if hasattr(event, 'repo') and event.repo and event.repo.full_name in plan_repo_names:
                    if event.type == 'PullRequestEvent':
                        if event.payload.get('action') == 'opened':
                            progress['in_progress_contributions'].append({
                                'type': 'pull_request',
                                'repository': event.repo.full_name,
                                'title': event.payload['pull_request']['title'],
                                'created_at': event.created_at
                            })
                        elif event.payload.get('action') == 'closed' and event.payload['pull_request'].get('merged'):
                            progress['completed_contributions'].append({
                                'type': 'pull_request_merged',
                                'repository': event.repo.full_name,
                                'title': event.payload['pull_request']['title'],
                                'merged_at': event.created_at
                            })
                            progress['metrics']['prs_merged'] += 1
                    
                    elif event.type == 'IssuesEvent' and event.payload.get('action') == 'closed':
                        progress['completed_contributions'].append({
                            'type': 'issue_closed',
                            'repository': event.repo.full_name,
                            'title': event.payload['issue']['title'],
                            'closed_at': event.created_at
                        })
                        progress['metrics']['issues_closed'] += 1
            
            # Calculate metrics
            total_planned = len(plan.opportunities)
            completed = len(progress['completed_contributions'])
            progress['metrics']['completion_rate'] = completed / total_planned if total_planned > 0 else 0
            progress['metrics']['repositories_contributed_to'] = len(set(
                contrib['repository'] for contrib in progress['completed_contributions'] + progress['in_progress_contributions']
            ))
            
        except Exception as e:
            print(f"Error tracking contributions: {e}")
        
        return progress
    
    def _mock_contribution_tracking(self) -> Dict[str, Any]:
        """Mock contribution tracking for demo"""
        return {
            'completed_contributions': [
                {
                    'type': 'pull_request_merged',
                    'repository': 'vinta/awesome-python',
                    'title': 'Add new ML library to list',
                    'merged_at': datetime.now() - timedelta(days=2)
                },
                {
                    'type': 'issue_closed',
                    'repository': 'scikit-learn/scikit-learn',
                    'title': 'Fix typo in documentation',
                    'closed_at': datetime.now() - timedelta(days=5)
                }
            ],
            'in_progress_contributions': [
                {
                    'type': 'pull_request',
                    'repository': 'pandas-dev/pandas',
                    'title': 'Improve error message in DataFrame.merge',
                    'created_at': datetime.now() - timedelta(days=1)
                }
            ],
            'metrics': {
                'completion_rate': 0.67,
                'hours_spent': 25,
                'repositories_contributed_to': 3,
                'prs_merged': 1,
                'issues_closed': 1
            }
        }
    
    def optimize_contribution_strategy(self, developer: DeveloperProfile, 
                                     past_contributions: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Optimize contribution strategy based on past performance"""
        analysis = {
            'performance_insights': [],
            'recommended_adjustments': [],
            'focus_areas': [],
            'skill_gaps': [],
            'next_level_opportunities': []
        }
        
        if not past_contributions:
            analysis['recommended_adjustments'].append("Start with documentation contributions to build confidence")
            analysis['focus_areas'].append("Learn contribution workflow and best practices")
            return analysis
        
        # Analyze contribution patterns
        contrib_types = [contrib.get('type', 'unknown') for contrib in past_contributions]
        type_counts = Counter(contrib_types)
        
        # Performance insights
        if type_counts.get('pull_request_merged', 0) > 0:
            analysis['performance_insights'].append("Successfully merging PRs shows good code quality")
        
        if type_counts.get('issue_closed', 0) > 0:
            analysis['performance_insights'].append("Closing issues demonstrates problem-solving skills")
        
        # Recommended adjustments
        if type_counts.get('pull_request_merged', 0) < 2:
            analysis['recommended_adjustments'].append("Focus on smaller, well-defined issues to increase merge success rate")
        
        if len(set(contrib.get('repository', '') for contrib in past_contributions)) < 3:
            analysis['recommended_adjustments'].append("Diversify contributions across more repositories")
        
        # Focus areas based on experience
        if developer.experience_level == 'beginner':
            analysis['focus_areas'].extend([
                "Documentation improvements",
                "Good first issues",
                "Testing and bug fixes"
            ])
        elif developer.experience_level == 'intermediate':
            analysis['focus_areas'].extend([
                "Feature implementation",
                "Code review participation",
                "Community engagement"
            ])
        else:
            analysis['focus_areas'].extend([
                "Architecture contributions",
                "Mentoring other contributors",
                "Project maintenance"
            ])
        
        # Skill gaps (technologies not yet contributed to)
        contributed_langs = set()
        for contrib in past_contributions:
            repo_name = contrib.get('repository', '')
            # This would normally look up the repository's language
            # For demo, we'll use a simple heuristic
            if 'python' in repo_name.lower():
                contributed_langs.add('Python')
            elif 'javascript' in repo_name.lower() or 'js' in repo_name.lower():
                contributed_langs.add('JavaScript')
        
        all_langs = set(developer.programming_languages)
        gap_langs = all_langs - contributed_langs
        analysis['skill_gaps'] = list(gap_langs)
        
        # Next level opportunities
        if len(past_contributions) >= 5:
            analysis['next_level_opportunities'].extend([
                "Consider becoming a maintainer",
                "Start your own open source project",
                "Speak at conferences about your contributions"
            ])
        elif len(past_contributions) >= 2:
            analysis['next_level_opportunities'].extend([
                "Take on larger, more complex issues",
                "Help review other contributors' PRs",
                "Write blog posts about your experience"
            ])
        
        return analysis
    
    def generate_contribution_report(self, plan: ContributionPlan, 
                                   progress: Dict[str, Any]) -> str:
        """Generate a comprehensive contribution report"""
        report = []
        report.append("📈 Open Source Contribution Report")
        report.append("=" * 50)
        
        # Developer summary
        dev = plan.developer
        report.append(f"\n👤 Developer: {dev.username}")
        report.append(f"📊 Experience Level: {dev.experience_level.title()}")
        report.append(f"💻 Languages: {', '.join(dev.programming_languages[:5])}")
        report.append(f"⏰ Weekly Availability: {dev.time_availability} hours")
        
        # Progress summary
        metrics = progress['metrics']
        report.append(f"\n📊 Progress Summary:")
        report.append(f"• Completion Rate: {metrics['completion_rate']:.1%}")
        report.append(f"• Hours Invested: {metrics['hours_spent']}")
        report.append(f"• Repositories: {metrics['repositories_contributed_to']}")
        report.append(f"• Merged PRs: {metrics['prs_merged']}")
        report.append(f"• Closed Issues: {metrics['issues_closed']}")
        
        # Impact estimation
        impact = plan.estimated_impact
        report.append(f"\n🎯 Estimated Impact:")
        report.append(f"• Total Impact Score: {impact.get('total_impact_score', 0):.2f}")
        report.append(f"• Learning Value: {impact.get('learning_value_score', 0):.2f}")
        report.append(f"• Potential User Reach: {impact.get('potential_user_reach', 0):,}")
        report.append(f"• Repository Diversity: {impact.get('repository_diversity', 0)}")
        
        # Top opportunities
        report.append(f"\n🚀 Top Contribution Opportunities:")
        for i, opp in enumerate(plan.opportunities[:5], 1):
            report.append(f"\n{i}. {opp.title}")
            report.append(f"   Repository: {opp.repository.name}")
            report.append(f"   Type: {opp.contribution_type.value.title()}")
            report.append(f"   Difficulty: {opp.difficulty.value.title()}")
            report.append(f"   Estimated Hours: {opp.estimated_hours}")
            report.append(f"   Match Score: {opp.match_score:.2f}")
        
        # Learning goals
        report.append(f"\n🎓 Learning Goals:")
        for goal in plan.learning_goals:
            report.append(f"• {goal}")
        
        # Timeline
        report.append(f"\n📅 Contribution Timeline:")
        for week, opportunities in list(plan.timeline.items())[:4]:  # Show first 4 weeks
            report.append(f"\n{week}:")
            for opp in opportunities:
                report.append(f"  • {opp.title} ({opp.estimated_hours}h)")
        
        return "\n".join(report)

In [ ]:
# Demo: Open Source Contribution Maximizer
def run_contribution_maximizer_demo():
    """Run the Open Source Contribution Maximizer demo"""
    print("🚀 Open Source Contribution Maximizer Demo")
    print("=" * 50)
    
    # Initialize the agent
    agent = OpenSourceContributionMaximizer()
    
    # Demo developer profile
    demo_username = "demo_developer"
    interests = ["machine learning", "web development", "data science"]
    preferred_languages = ["Python", "JavaScript", "TypeScript"]
    
    try:
        # Find contribution opportunities
        print(f"\n🔍 Finding opportunities for {demo_username}...")
        plan = agent.find_contribution_opportunities(
            developer_username=demo_username,
            interests=interests,
            preferred_languages=preferred_languages
        )
        
        print(f"\n✅ Generated plan with {len(plan.opportunities)} opportunities")
        
        # Track contributions (using mock data)
        print("\n📊 Tracking contribution progress...")
        progress = agent.track_contributions(demo_username, plan)
        
        # Optimize strategy
        print("\n🎯 Optimizing contribution strategy...")
        optimization = agent.optimize_contribution_strategy(
            plan.developer, 
            progress['completed_contributions'] + progress['in_progress_contributions']
        )
        
        # Generate comprehensive report
        print("\n📋 Generating contribution report...")
        report = agent.generate_contribution_report(plan, progress)
        
        # Display results
        print("\n" + "=" * 60)
        print(report)
        
        print("\n" + "=" * 60)
        print("🎯 Strategy Optimization Results:")
        print("\n📈 Performance Insights:")
        for insight in optimization['performance_insights']:
            print(f"• {insight}")
        
        print("\n🔧 Recommended Adjustments:")
        for adjustment in optimization['recommended_adjustments']:
            print(f"• {adjustment}")
        
        print("\n🎯 Focus Areas:")
        for area in optimization['focus_areas']:
            print(f"• {area}")
        
        if optimization['skill_gaps']:
            print("\n📚 Skill Gaps to Address:")
            for gap in optimization['skill_gaps']:
                print(f"• {gap}")
        
        print("\n🚀 Next Level Opportunities:")
        for opportunity in optimization['next_level_opportunities']:
            print(f"• {opportunity}")
        
        # Advanced analytics
        print("\n" + "=" * 60)
        print("📊 Advanced Analytics:")
        
        # Contribution type distribution
        contrib_types = [opp.contribution_type.value for opp in plan.opportunities]
        type_counts = Counter(contrib_types)
        print("\n📈 Contribution Type Distribution:")
        for contrib_type, count in type_counts.most_common():
            percentage = (count / len(plan.opportunities)) * 100
            print(f"• {contrib_type.title()}: {count} ({percentage:.1f}%)")
        
        # Difficulty distribution
        difficulties = [opp.difficulty.value for opp in plan.opportunities]
        diff_counts = Counter(difficulties)
        print("\n⚡ Difficulty Distribution:")
        for difficulty, count in diff_counts.most_common():
            percentage = (count / len(plan.opportunities)) * 100
            print(f"• {difficulty.title()}: {count} ({percentage:.1f}%)")
        
        # Repository statistics
        repo_stats = {
            'total_stars': sum(opp.repository.stars for opp in plan.opportunities),
            'avg_stars': np.mean([opp.repository.stars for opp in plan.opportunities]),
            'total_contributors': sum(opp.repository.contributors for opp in plan.opportunities),
            'avg_responsiveness': np.mean([opp.repository.maintainer_responsiveness for opp in plan.opportunities])
        }
        
        print("\n⭐ Repository Statistics:")
        print(f"• Total Stars Across Repos: {repo_stats['total_stars']:,}")
        print(f"• Average Stars per Repo: {repo_stats['avg_stars']:,.0f}")
        print(f"• Total Contributors: {repo_stats['total_contributors']:,}")
        print(f"• Average Maintainer Responsiveness: {repo_stats['avg_responsiveness']:.2f}")
        
        print("\n✅ Demo completed successfully!")
        
        return plan, progress, optimization
        
    except Exception as e:
        print(f"❌ Error in demo: {e}")
        return None, None, None

# Run the demo
if __name__ == "__main__":
    plan, progress, optimization = run_contribution_maximizer_demo()

## Production Deployment with Celery

### Celery Task Definitions

In [ ]:
# Celery tasks for distributed processing
try:
    from celery import Celery
    
    # Initialize Celery app
    celery_app = Celery('os_contribution_maximizer')
    celery_app.config_from_object({
        'broker_url': 'redis://localhost:6379/0',
        'result_backend': 'redis://localhost:6379/0',
        'task_serializer': 'json',
        'accept_content': ['json'],
        'result_serializer': 'json',
        'timezone': 'UTC',
        'enable_utc': True,
    })
    
    @celery_app.task(bind=True, max_retries=3)
    def analyze_repository_task(self, repo_url: str, github_token: str = None):
        """Celery task to analyze a repository"""
        try:
            analyzer = GitHubAnalyzer(github_token)
            # Extract repo name from URL
            repo_name = repo_url.split('github.com/')[-1]
            
            if analyzer.github:
                repo = analyzer.github.get_repo(repo_name)
                repository = analyzer._analyze_repository(repo)
                
                return {
                    'success': True,
                    'repository': {
                        'name': repository.name,
                        'full_name': repository.full_name,
                        'description': repository.description,
                        'language': repository.language,
                        'stars': repository.stars,
                        'forks': repository.forks,
                        'open_issues': repository.open_issues,
                        'newcomer_friendliness': repository.newcomer_friendliness,
                        'activity_score': repository.activity_score
                    }
                }
            else:
                return {'success': False, 'error': 'GitHub token not available'}
                
        except Exception as exc:
            if self.request.retries < self.max_retries:
                # Exponential backoff
                countdown = 2 ** self.request.retries
                raise self.retry(countdown=countdown, exc=exc)
            return {'success': False, 'error': str(exc)}
    
    @celery_app.task(bind=True, max_retries=3)
    def find_opportunities_task(self, developer_data: Dict[str, Any], repositories_data: List[Dict[str, Any]]):
        """Celery task to find contribution opportunities"""
        try:
            # Reconstruct objects from data
            developer = DeveloperProfile(**developer_data)
            repositories = [Repository(**repo_data) for repo_data in repositories_data]
            
            matcher = OpportunityMatcher()
            opportunities = matcher.find_opportunities(developer, repositories)
            
            # Serialize opportunities
            serialized_opps = []
            for opp in opportunities:
                serialized_opps.append({
                    'title': opp.title,
                    'description': opp.description,
                    'contribution_type': opp.contribution_type.value,
                    'difficulty': opp.difficulty.value,
                    'estimated_hours': opp.estimated_hours,
                    'match_score': opp.match_score,
                    'repository_name': opp.repository.name
                })
            
            return {'success': True, 'opportunities': serialized_opps}
            
        except Exception as exc:
            if self.request.retries < self.max_retries:
                countdown = 2 ** self.request.retries
                raise self.retry(countdown=countdown, exc=exc)
            return {'success': False, 'error': str(exc)}
    
    @celery_app.task
    def daily_opportunity_scan(github_token: str = None):
        """Daily task to scan for new opportunities"""
        try:
            analyzer = GitHubAnalyzer(github_token)
            
            # Search for trending repositories with good first issues
            trending_repos = analyzer.search_repositories(
                query="good first issue created:>2024-01-01",
                min_stars=100,
                max_results=50
            )
            
            # Store results (would typically save to database)
            results = {
                'scan_date': datetime.now().isoformat(),
                'repositories_found': len(trending_repos),
                'total_opportunities': sum(len(repo.good_first_issues) + len(repo.help_wanted_issues) for repo in trending_repos)
            }
            
            return results
            
        except Exception as e:
            return {'error': str(e)}
    
    print("✅ Celery tasks defined successfully")
    print("📚 Usage:")
    print("  # Start Celery worker:")
    print("  celery -A open_source_maximizer worker --loglevel=info")
    print("  ")
    print("  # Start periodic tasks:")
    print("  celery -A open_source_maximizer beat --loglevel=info")
    print("  ")
    print("  # Monitor tasks:")
    print("  celery -A open_source_maximizer flower")
    
except ImportError:
    print("⚠️ Celery not available. Install with: pip install celery redis")
    print("For production deployment, also set up Redis server.")

## Production Deployment Considerations

### Infrastructure Requirements
- **Redis**: Message broker for Celery task queue
- **PostgreSQL**: Store developer profiles, repositories, and contribution history
- **Elasticsearch**: Full-text search for repositories and issues
- **Docker**: Containerized deployment

### API Integration
- **GitHub API**: Repository analysis and developer profiling
- **GitLab API**: Support for GitLab repositories
- **Bitbucket API**: Support for Bitbucket repositories
- **NPM/PyPI APIs**: Package ecosystem analysis

### Advanced Features
- **Machine Learning**: Improved matching algorithms using collaborative filtering
- **Social Network Analysis**: Identify influential contributors and projects
- **Contribution Impact Tracking**: Measure real-world impact of contributions
- **Automated PR Generation**: Generate starter code for issues
- **Mentorship Matching**: Connect newcomers with experienced contributors

### Scaling Considerations
- **Rate Limiting**: Respect GitHub API rate limits
- **Caching**: Cache repository and user data
- **Load Balancing**: Distribute API requests across multiple workers
- **Data Pipeline**: Continuous updates of repository and contribution data

### Monitoring and Analytics
- **Contribution Success Rates**: Track which recommendations lead to successful contributions
- **Developer Progression**: Monitor skill development over time
- **Repository Health**: Track maintainer responsiveness and community activity
- **Impact Metrics**: Measure ecosystem-wide contribution improvements